In [1]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

Current Project Root Directory set to: /Users/luca/Documents/GitHub/US_conflict_forecasting
Current Search Path set to: ['/Users/luca/Documents/GitHub/US_conflict_forecasting', '/Users/luca/Documents/GitHub/US_conflict_forecasting/2_Dataset/c_Merge_Cleaning', '/Users/luca/anaconda3/envs/ML_Lab/lib/python312.zip', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/lib-dynload', '', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages', '/Users/luca/Documents/GitHub/US_conflict_forecasting']


In [2]:
import pandas as pd

In [3]:
# Load the dataset that was crated in 'all_zips' notebook (12_zip_ACS.csv).
# This holds all ZIP codes, 12 times each, with all ACS data.

final_12_ACS = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/a_Input_Datasets/12_ZCTA_ACS.csv'))


# Load conlfict data from ACLED, with all 12 months for each ZIP code
# IMPORTANT: This only holds ZIPs that are present in ACLED dataset, meaning they had at least one conflict in at least one month.
# If there was no conflict at all in one year in a specific ZIP, it is missing.
# The aim of this script is to get each missing ZIP code into the the set and then assign 0s for not being present in ACLED.

conflict = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/ConflictDataClean.csv'))

In [4]:
# Create list of all ZIP codes, taken from the ACS dataset.
zip_list = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/a_Input_Datasets/12_ZCTA_ACS.csv'))['ZCTA'].tolist()


In [5]:
# Create a list of the ZIPs that are missing from the conflict set, by literally subtracting the two sets from each other.

missing_zip = list(set(zip_list) - set(conflict['ZCTA']))

# Create it into df
missing_zip = pd.DataFrame(missing_zip, columns=['ZCTA'])

# Now, apply same logic as 'all_zips' notebook to create a dataset with all ZIPs, 12 times each. FOR EACH MISSING ZIP
months = pd.date_range(start='2022-01-01', end='2022-12-01', freq='MS').strftime('%Y-%m').tolist()

monthly_missing = pd.DataFrame()

for index, row in missing_zip.iterrows():
    zip_code = row['ZCTA']
    zip_code_entries = pd.DataFrame({'ZCTA': [zip_code] * len(months), 'Month': months})
    monthly_missing = pd.concat([monthly_missing, zip_code_entries])


In [6]:
# Full outer merge the missing (x12) ZIPs with the conflict data.
combined_df = pd.merge(conflict, monthly_missing, on='ZCTA', how='outer')

In [10]:
# Fix month columns
combined_df['month'] = combined_df['event_MY'].fillna(combined_df['Month'])
# Drop the old columns
combined_df.drop(['event_MY', 'Month'], axis=1, inplace=True)
# Reorder the variable order
combined_df = combined_df[['ZCTA', 'month', 'admin2', 'fatalities', 'event_count']]
# Fill nans from merge to 0 (not present in ACLED, no conflict)
combined_df.fillna(0, inplace=True)

In [12]:
#Converting to csv + saving in Folder: b_Partial_Datasets
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets')
output_filename_1 = "conflict_all_ZCTA.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

combined_df.to_csv(output_path_1, index=False)